In [1]:
# Install dependencies
%pip install numpy matplotlib Pillow


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\sulta\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# Use non-interactive backend to reduce memory overhead
import matplotlib
matplotlib.use("Agg")


In [3]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np


In [4]:
# Configuration (edit these)
@dataclass
class SynthConfig:
    out_dir: str = "Synthetic_dataset"
    num_images: int = 3000
    seed: int = 123
    width_px: int = 1600
    height_px: int = 900
    dpi: int = 200
    # If you hit MemoryError, reduce width_px/height_px or dpi and rerun.
    min_lines: int = 1
    max_lines: int = 5
    min_points: int = 6
    max_points: int = 12
    line_styles: tuple = ("-", "--", ":", "-.")
    markers: tuple = ("o", "s", "^", "D", "v", "P", "X")
    colors: tuple = (
        "#e41a1c",
        "#377eb8",
        "#4daf4a",
        "#984ea3",
        "#ff7f00",
        "#a65628",
        "#f781bf",
        "#999999",
    )
    background_colors: tuple = ("#ffffff", "#f7f7f7", "#f2f2f2", "#faf9f5")

config = SynthConfig()
config


SynthConfig(out_dir='Synthetic_dataset', num_images=3000, seed=123, width_px=1600, height_px=900, dpi=200, min_lines=1, max_lines=5, min_points=6, max_points=12, line_styles=('-', '--', ':', '-.'), markers=('o', 's', '^', 'D', 'v', 'P', 'X'), colors=('#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#a65628', '#f781bf', '#999999'), background_colors=('#ffffff', '#f7f7f7', '#f2f2f2', '#faf9f5'))

In [5]:
# Save config for reproducibility
out_dir = Path(config.out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "config.json").write_text(json.dumps(asdict(config), indent=2), encoding="utf-8")
out_dir


WindowsPath('Synthetic_dataset')

In [6]:
def _data_to_image_coords(ax, xy, fig_height_px):
    x_disp, y_disp = ax.transData.transform(xy)
    return float(x_disp), float(fig_height_px - y_disp)

def _add_axis_labels(points, ax, fig_height_px):
    x_min, x_max = ax.get_xlim()
    y_min, y_max = ax.get_ylim()

    x_min_px, y_min_px = _data_to_image_coords(ax, (x_min, y_min), fig_height_px)
    x_max_px, _ = _data_to_image_coords(ax, (x_max, y_min), fig_height_px)
    _, y_max_px = _data_to_image_coords(ax, (x_min, y_max), fig_height_px)

    points.extend([
        {"x": x_min_px, "y": y_min_px, "label": "ymin", "topBarPixelDistance": 0, "bottomBarPixelDistance": 0, "deviationPixelDistance": 0},
        {"x": x_min_px, "y": y_max_px, "label": "ymax", "topBarPixelDistance": 0, "bottomBarPixelDistance": 0, "deviationPixelDistance": 0},
        {"x": x_min_px, "y": y_min_px, "label": "xmin", "topBarPixelDistance": 0, "bottomBarPixelDistance": 0, "deviationPixelDistance": 0},
        {"x": x_max_px, "y": y_min_px, "label": "xmax", "topBarPixelDistance": 0, "bottomBarPixelDistance": 0, "deviationPixelDistance": 0},
    ])

def generate_plot(rng, np_rng, cfg: SynthConfig):
    fig_width = cfg.width_px / cfg.dpi
    fig_height = cfg.height_px / cfg.dpi

    fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=cfg.dpi)
    fig.patch.set_facecolor(rng.choice(cfg.background_colors))
    ax.set_facecolor(rng.choice(cfg.background_colors))

    num_lines = rng.randint(cfg.min_lines, cfg.max_lines)
    num_points = rng.randint(cfg.min_points, cfg.max_points)

    x_values = np.linspace(0, rng.randint(8, 36), num_points)
    line_outputs = []

    for idx in range(num_lines):
        base = rng.uniform(0.5, 2.5)
        trend = rng.uniform(-0.1, 0.2)
        noise = rng.uniform(0.05, 0.3)
        noise_vec = np_rng.normal(0, noise, size=num_points) * np.linspace(1, 1.2, num_points)
        y_values = base + trend * x_values + noise_vec
        y_values = np.clip(y_values + rng.uniform(4, 9), 0, None)

        yerr_up = np_rng.uniform(0.1, 0.8, size=num_points)
        yerr_down = np_rng.uniform(0.1, 0.8, size=num_points)

        line_style = rng.choice(cfg.line_styles)
        marker = rng.choice(cfg.markers)
        color = cfg.colors[idx % len(cfg.colors)]
        line_name = f"Line_{idx + 1}"

        ax.errorbar(
            x_values,
            y_values,
            yerr=[yerr_down, yerr_up],
            fmt=marker,
            linestyle=line_style,
            color=color,
            capsize=rng.randint(3, 7),
            linewidth=rng.uniform(1.5, 2.8),
            markersize=rng.uniform(5, 8),
            label=line_name,
        )

        line_outputs.append({
            "lineName": line_name,
            "points": [],
            "_y_values": y_values,
            "_x_values": x_values,
            "_yerr_up": yerr_up,
            "_yerr_down": yerr_down,
        })

    ax.set_xlabel(rng.choice(["Time on Study (Months)", "Dose (mg)", "Day", "Week"]), fontweight="bold")
    ax.set_ylabel(rng.choice(["Response", "HbA1c (%)", "Concentration", "Score"]), fontweight="bold")

    if rng.random() > 0.3:
        ax.grid(True, linestyle=rng.choice([":", "--", "-."]), alpha=0.3)

    ax.legend(loc="upper left", frameon=False)
    ax.set_xlim(min(x_values) - 1, max(x_values) + 1)

    fig.canvas.draw()
    fig_height_px = fig.get_size_inches()[1] * fig.dpi

    for line in line_outputs:
        for x_val, y_val, y_up, y_down in zip(
            line["_x_values"], line["_y_values"], line["_yerr_up"], line["_yerr_down"]
        ):
            x_px, y_px = _data_to_image_coords(ax, (x_val, y_val), fig_height_px)
            _, y_up_px = _data_to_image_coords(ax, (x_val, y_val + y_up), fig_height_px)
            _, y_down_px = _data_to_image_coords(ax, (x_val, y_val - y_down), fig_height_px)

            line["points"].append({
                "x": x_px,
                "y": y_px,
                "label": "",
                "topBarPixelDistance": max(0.0, y_px - y_up_px),
                "bottomBarPixelDistance": max(0.0, y_down_px - y_px),
                "deviationPixelDistance": max(y_px - y_up_px, y_down_px - y_px),
            })

        _add_axis_labels(line["points"], ax, fig_height_px)
        line.pop("_x_values")
        line.pop("_y_values")
        line.pop("_yerr_up")
        line.pop("_yerr_down")

    return fig, line_outputs


In [7]:
# Preview a few samples before full generation
preview_dir = Path(config.out_dir) / "preview"
preview_dir.mkdir(parents=True, exist_ok=True)

preview_rng = random.Random(config.seed)
preview_np_rng = np.random.default_rng(config.seed)

for i in range(3):
    fig, _ = generate_plot(preview_rng, preview_np_rng, config)
    fig.savefig(preview_dir / f"preview_{i}.png")
    plt.close(fig)

sorted(preview_dir.glob("*.png"))


[WindowsPath('Synthetic_dataset/preview/preview_0.png'),
 WindowsPath('Synthetic_dataset/preview/preview_1.png'),
 WindowsPath('Synthetic_dataset/preview/preview_2.png')]

In [8]:
# Generate dataset (memory-safe)
import gc

rng = random.Random(config.seed)
np_rng = np.random.default_rng(config.seed)

images_dir = Path(config.out_dir) / "images"
labels_dir = Path(config.out_dir) / "labels"
images_dir.mkdir(parents=True, exist_ok=True)
labels_dir.mkdir(parents=True, exist_ok=True)

# Control memory by periodically forcing garbage collection
gc_interval = 25

for idx in range(config.num_images):
    fig, label_lines = generate_plot(rng, np_rng, config)

    filename = f"synthetic_{idx:04d}.png"
    image_path = images_dir / filename
    label_path = labels_dir / f"synthetic_{idx:04d}.json"

    fig.savefig(image_path)
    fig.clf()
    plt.close(fig)

    with label_path.open("w", encoding="utf-8") as handle:
        json.dump(
            [{"label": {"lineName": line["lineName"]}, "points": line["points"]} for line in label_lines],
            handle,
            indent=2,
        )

    if (idx + 1) % gc_interval == 0:
        gc.collect()

print(f"Generated {config.num_images} images in {config.out_dir}")


Generated 3000 images in Synthetic_dataset


In [9]:
# Quick sanity check
images = sorted(images_dir.glob("*.png"))
labels = sorted(labels_dir.glob("*.json"))
len(images), len(labels)


(3000, 3000)